# NoCapAI — Data Science Model Training

**Objective:** Predict whether a library loan is likely to become overdue.

This notebook demonstrates the complete academic workflow: EDA → preprocessing → model comparison → evaluation → final model serialization.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
DATA = Path("../data/training_data.csv")
df = pd.read_csv(DATA)
print("Shape:", df.shape)
df.head()

## 1. Data Quality & Target Distribution

In [ ]:
print(df.info())
print("\nMissing values:\n", df.isna().sum())
print("\nTarget balance:\n", df["overdue"].value_counts(normalize=True).round(3))

In [ ]:
df["overdue"].value_counts().sort_index().plot(kind="bar", title="Target Distribution", xlabel="Overdue (0/1)", ylabel="Records")
plt.tight_layout()
plt.show()

## 2. Feature Engineering / Preprocessing

Numeric variables are standardised and categorical variables are one-hot encoded. The preprocessing is placed inside a scikit-learn Pipeline to prevent train/test leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

features=[c for c in df.columns if c!="overdue"]
X=df[features]; y=df["overdue"]
cat=["member_type","genre"]; num=[c for c in features if c not in cat]
pre=ColumnTransformer([("numeric",StandardScaler(),num),("categorical",OneHotEncoder(handle_unknown="ignore"),cat)])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=42)

## 3. Model Comparison

In [ ]:
models={
"Logistic Regression":LogisticRegression(max_iter=1500,class_weight="balanced",random_state=42),
"Decision Tree":DecisionTreeClassifier(max_depth=6,class_weight="balanced",random_state=42),
"Random Forest":RandomForestClassifier(n_estimators=300,max_depth=10,min_samples_leaf=2,class_weight="balanced",random_state=42,n_jobs=-1),
"Gradient Boosting":GradientBoostingClassifier(random_state=42)}
rows=[]
for name,model in models.items():
    pipe=Pipeline([("preprocess",pre),("model",model)])
    pipe.fit(X_train,y_train)
    pred=pipe.predict(X_test); prob=pipe.predict_proba(X_test)[:,1]
    rows.append([name,accuracy_score(y_test,pred),precision_score(y_test,pred,zero_division=0),recall_score(y_test,pred,zero_division=0),f1_score(y_test,pred,zero_division=0),roc_auc_score(y_test,prob)])
comparison=pd.DataFrame(rows,columns=["Model","Accuracy","Precision","Recall","F1","ROC-AUC"]).sort_values("ROC-AUC",ascending=False)
comparison.round(3)

## 4. Final Model — Random Forest

Random Forest is retained as the final model for the application because it models non-linear relationships, works well with mixed features after encoding, and supports feature-importance explanations. The model is an academic prototype; synthetic data must not be presented as real-world validation.

In [ ]:
final_model=Pipeline([("preprocess",pre),("model",models["Random Forest"])])
final_model.fit(X_train,y_train)
pred=final_model.predict(X_test); prob=final_model.predict_proba(X_test)[:,1]
print(classification_report(y_test,pred,target_names=["On time","Overdue"],zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=["On time","Overdue"])
plt.tight_layout(); plt.show()

## 5. Cross-Validation

Five-fold stratified cross-validation is reported as an additional stability check.

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scores=cross_val_score(final_model,X,y,cv=cv,scoring="roc_auc")
print("Fold ROC-AUC:", scores.round(3))
print("Mean ROC-AUC:", round(scores.mean(),3))

## 6. Conclusion

The deployed model predicts overdue risk from borrower history, current load, book popularity, loan duration, member type and genre. The API serves the trained pipeline so preprocessing and inference remain identical. For a stronger research submission, replace the synthetic dataset with anonymised historical transactions and repeat the full evaluation.